In [1]:
import ROOT
import pandas as pd
import numpy as np
import os
import sys
import ipynbname
from pathlib import Path

project_root = str(ipynbname.path().parent.parent)
sys.path.append(project_root)
project_root=Path(project_root)

from processing import SiphraAcquisition, MetadataLoader
from analysis import fit_peak_expbg

# Constants
BITS12 = 2**12
BITS9 = 2**9 # 512 typical number of bins used

# Energy emission peaks in MeV
Cs137_MeV = 0.662
Na22_MeV = [0.511, 1.275, 1.786]
Co60_MeV = [1.173, 1.332, 2,505]
Am241_MeV = 0.0595
# Background emission
K40_MeV = 1.460
Tl208_MEV = 2.614

colors = [ROOT.kRed, ROOT.kBlue, ROOT.kGreen, ROOT.kOrange, ROOT.kViolet, ROOT.kYellow, ROOT.kSpring, ROOT.kCyan,]

In [2]:
files = sorted((project_root/'data/260323').glob('SUBT_*'))
acqs = [SiphraAcquisition(file) for file in files]
print(acqs)

[SIPHRAAcquisition(File: 'SUBT_01_EnergyResolution_thr30_gain1over100_1over3_Background'), SIPHRAAcquisition(File: 'SUBT_02_EnergyResolution_thr30_gain1over100_1over3_Cs137'), SIPHRAAcquisition(File: 'SUBT_03_EnergyResolution_thr30_gain1over100_1over3_Background2'), SIPHRAAcquisition(File: 'SUBT_04_EnergyResolution_thr30_gain1over100_1over3_Na22'), SIPHRAAcquisition(File: 'SUBT_05_EnergyResolution_thr30_gain1over100_1over3_Background4'), SIPHRAAcquisition(File: 'SUBT_06_EnergyResolution_thr30_gain1over100_1over3_Co60'), SIPHRAAcquisition(File: 'SUBT_07_EnergyResolution_thr30_gain1over100_1over3_Background6'), SIPHRAAcquisition(File: 'SUBT_08_EnergyResolution_thr30_gain1over100_1over3_Am241'), SIPHRAAcquisition(File: 'SUBT_09_EnergyResolution_thr30_gain1over100_1over3_Background8')]


In [3]:
# histograms
hists = {}
sources = []
for sgnl, bg in zip(acqs[1::2], acqs[2::2]):
    filepath = sgnl.filepath.stem
    src = (MetadataLoader.load(sgnl.metadataFile)).source
    sources.append(src)
    print(src)
    # Signal and Background
    hist_sgnl = ROOT.TH1F(f"{src} Signal", "", BITS12, 0, BITS12)
    hist_bg = ROOT.TH1F(f"{src} Background", "", BITS12, 0 , BITS12)
    hist_sgnl.Fill(sgnl['s']/len(sgnl.active_chs))
    hist_bg.Fill(bg['s']/len(bg.active_chs))
    hist_bg.Scale(sgnl.exposure/bg.exposure)
    # Clean spectrum
    hist_clean = hist_sgnl.Clone(f"{src} Clean")
    hist_clean.Add(hist_bg, -1)
    for hist in [hist_sgnl, hist_bg, hist_clean]:
        # hist.SetTitle(r"^{22}Na spectra - CI gain = 1/0.75 pC")
        hist.GetXaxis().SetTitle("ADC channel number")
        hist.GetYaxis().SetTitle(r"Normalized counts")
    hists[src] = hist_sgnl
    hists[f"{src}_BG"] = hist_bg
    hists[f"{src}_clean"] = hist_clean
    del hist_sgnl, hist_bg
#print(hists)

Cs-137
Na22
Co-60
Am-241


In [4]:
if ROOT.gROOT.FindObject('cv'):
    ROOT.gROOT.FindObject('cv').Close()

Yinit = 0.82 # For stat boxes

cv = ROOT.TCanvas('cv', 'cv', 1600, 1200)
cv.Divide(2,2)

ROOT.gStyle.SetOptStat(11)
ROOT.gStyle.SetStatFontSize(0.03)
ROOT.gStyle.SetStatW(0.16)

lgds = [ROOT.TLegend(0.48, 0.61, 0.75, 0.83) for _ in range(4)]

for idx, (src, lg) in enumerate(zip(sources, lgds)):   
    cv.cd(idx+1)
    
    sgnl = hists[src]
    bg = hists[src+'_BG']
    clean = hists[src+'_clean']
    
    lg.AddEntry(sgnl, "Signal")
    lg.AddEntry(bg, "Background")
    lg.AddEntry(clean, "Subtracted")
    sgnl.SetLineColor(colors[0])
    bg.SetLineColor(colors[1])
    clean.SetLineColor(colors[2])
    sgnl.SetTitle(src)
    sgnl.Draw("hist")
    bg.Draw("sames hist")
    clean.Draw("sames hist")
    ROOT.gPad.Update()
    for i, sp in enumerate([sgnl, bg, clean]):
        st = sp.FindObject("stats")
        # print(type(st))
        st.SetY1NDC(Yinit - i*0.08)
        st.SetY2NDC(0.1)
    lg.Draw()
    cv.cd(idx+1).SetLogy()
    cv.Draw()




In [5]:
#Fitting all the visible peaks. Values around peak input manually

Cs137_fit = ROOT.TF1("Cs137_fit", "gaus", 120,250)

Co60a_fit = ROOT.TF1("Co60a_fit", "gaus", 200,350)
Co60b_fit = ROOT.TF1("Co60b_fit", "gaus", 450,500)

Na22a_fit = ROOT.TF1("Na22a_fit", "gaus", 150,200)
Na22b_fit = ROOT.TF1("Na22b_fit", "gaus", 320,420)
Na22c_fit = ROOT.TF1("Na22c_fit", "gaus", 450,520)

fits = [Cs137_fit, Co60a_fit, Co60b_fit, Na22a_fit, Na22b_fit, Na22c_fit]

#Fit to histogram and find mean value of peak
hists['Cs-137_clean'].Fit(Cs137_fit, "R+S")
hists['Co-60_clean'].Fit(Co60a_fit, "R+S")
hists['Co-60_clean'].Fit(Co60b_fit, "R+S")
hists['Na22_clean'].Fit(Na22a_fit, "R+S")
hists['Na22_clean'].Fit(Na22b_fit, "R+S")
hists['Na22_clean'].Fit(Na22c_fit, "R+S")

Cs137_mean = Cs137_fit.GetParameter('Mean')
Co60a_mean = Co60a_fit.GetParameter('Mean')
Co60b_mean = Co60b_fit.GetParameter('Mean')
Na22a_mean = Na22a_fit.GetParameter('Mean')
Na22b_mean = Na22b_fit.GetParameter('Mean')
Na22c_mean = Na22c_fit.GetParameter('Mean')

channels = [Na22a_mean, Na22b_mean, Na22c_mean]
energies = Na22_MeV
channels = np.array(channels, dtype='float64')
energies = np.array(energies, dtype='float64')

g = ROOT.TGraph(len(channels), channels, energies)

f = ROOT.TF1("calib", "pol1", min(channels), max(channels))
g.Fit(f)

a = f.GetParameter(1)
b = f.GetParameter(0)
print(a)
print(b)








ROOT.gPad.Modified()
ROOT.gPad.Update()

#lg.Draw()
#cv.Draw()

0.004280158214429103
-0.2655932438428593
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      196.518
NDf                       =          127
Edm                       =  2.29967e-07
NCalls                    =           65
Constant                  =      2459.66   +/-   7.74946     
Mean                      =      183.993   +/-   0.078652    
Sigma                     =      29.2438   +/-   0.0713573    	 (limited)
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      356.977
NDf                       =          147
Edm                       =  1.20533e-06
NCalls                    =           64
Constant                  =      1361.97   +/-   6.38965     
Mean                      =       275.54   +/-   0.144272    
Sigma                     =      32.7382   +/-   0.141925     	 (limited)
****************************************
Minimizer is Minuit2 / Migrad
Chi2                  

In [6]:
if ROOT.gROOT.FindObject('cv_cal'):
    ROOT.gROOT.FindObject('cv_cal').Close()


    
hist = hists['Na22_clean']

emin = a * hist.GetXaxis().GetXmin() + b
emax = a * hist.GetXaxis().GetXmax() + b

emax_2 = a * BITS12 + b
emin_2 = a * 0 + b

print(emin)
print(emin_2)
print(emax)
print(emax_2)

print(acqs[3])
print(acqs[4])

data_bg_clb = a * (acqs[4]['s']/len(acqs[4].active_chs)) + b
data_src_clb = a * (acqs[3]['s']/len(acqs[4].active_chs)) + b

print(data_bg_clb)

print(data_bg_clb.max())
print(data_bg_clb.min())

h_cal = ROOT.TH1F("h_cal", "Calibrated Spectrum",
             hist.GetNbinsX(), emin, emax)

print(hist.GetNbinsX())
print(BITS12)

#for i in range(1, hist.GetNbinsX() + 1):
#    counts = hist.GetBinContent(i)
#    h_cal.SetBinContent(i, counts)

hist_bg_clb = ROOT.TH1F("Calibrated background", "Calibrated Histograms", BITS12, emin, emax)
hist_src_clb = ROOT.TH1F("Calibrated signal", "Calibrated Histograms", BITS12, emin, emax)


hist_bg_clb.Fill(data_bg_clb)
#hist_bg_clb.Scale(NORMFACTOR)

hist_src_clb.Fill(data_src_clb)

hist_clean_clb = hist_src_clb.Clone("Calibrated signal no BG")
hist_clean_clb.Add(hist_bg_clb, -1)

#for hist, clr in zip([hist_bg_clb, hist_src_clb, hist_clean_clb], colors):
#    hist.GetXaxis().SetTitle("Energy (MeV)")
 #   hist.GetYaxis().SetTitle("Normalized counts")
 #   hist.SetLineColor(clr)  
#
#cv_cal.Draw()

-0.2655932438428593
-0.2655932438428593
17.265934802458748
17.265934802458748
SIPHRAAcquisition(File: 'SUBT_04_EnergyResolution_thr30_gain1over100_1over3_Na22')
SIPHRAAcquisition(File: 'SUBT_05_EnergyResolution_thr30_gain1over100_1over3_Background4')
[0.96495224 0.7338237  0.70760773 ... 0.76057469 0.40532156 0.49787998]
16.08942631326755
-0.11364762723062616
4096
4096


True

In [7]:
# Plot all the histograms
if ROOT.gROOT.FindObject('cv1'):
    ROOT.gROOT.FindObject('cv1').Close()
cv1 = ROOT.TCanvas("cv1", "cv1", 1600, 800)
cv1.Divide(2,1)

lg1 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
cv1.cd(1)
hist_bg_clb.SetLineColor(colors[0])
hist_src_clb.SetLineColor(colors[1])
lg1.AddEntry(hist_bg_clb, "Background", "l")
lg1.AddEntry(hist_src_clb, r"Signal ^{137}Cs", "l")
hist_bg_clb.Draw("hist")
hist_src_clb.Draw("sames hist")
lg1.Draw()
cv1.cd(1).SetLogy()
hist_bg_clb.GetYaxis().SetRangeUser(0,1e4)


# lg2 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
cv1.cd(2)
hist_clean_clb.SetLineColor(colors[2])
hist_clean_clb.Draw("hist")
cv1.cd(2).SetLogy()

cv1.Draw()

Error in <THistPainter::PaintInit>: Cannot set Y axis to log scale


In [8]:
# Plot all the histograms




Na22a_fit_cal = ROOT.TF1("Na22a_fit_cal", "gaus", 0.3, 0.6)
Na22b_fit_cal = ROOT.TF1("Na22b_fit_cal", "gaus", 1.1 , 1.5)
Na22c_fit_cal = ROOT.TF1("Na22c_fit_cal", "gaus", 1.5, 1.9)
hist_clean_clb.Fit(Na22a_fit_cal, "R+S")
hist_clean_clb.Fit(Na22b_fit_cal, "R+S")
hist_clean_clb.Fit(Na22c_fit_cal, "R+S")

hist_clean_clb.Draw("hist")
cv1.cd(idx+1).SetLogy()

ROOT.gPad.Modified()
ROOT.gPad.Update()



Error in <THistPainter::PaintInit>: Cannot set Y axis to log scale


****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      137.425
NDf                       =           67
Edm                       =  2.14133e-06
NCalls                    =           84
Constant                  =      190.012   +/-   2.40871     
Mean                      =     0.492756   +/-   0.00341755  
Sigma                     =     0.142388   +/-   0.00438193   	 (limited)
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      465.001
NDf                       =           91
Edm                       =  1.22703e-07
NCalls                    =           75
Constant                  =       611.78   +/-   3.97344     
Mean                      =      1.31966   +/-   0.00146494  
Sigma                     =     0.178741   +/-   0.0023569    	 (limited)
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      742.699
NDf                   

In [9]:
def calibration_fit(histogram, energy_ranges, energies):
    """Function to create a linear calibration fit based on a histogram. Returns slope and constant of linear fit. 
    Inputs: Histogram to base calibration on.
    Ranges within which the peaks of the histogram are located.
    Known energies of these peaks, in MeV."""
    
    channels = []
    for i in range(len(energy_ranges)):
        cal_fit=ROOT.TF1("cal_fit_" + str(i), "gaus", energy_ranges[i][0], energy_ranges[i][1])
        hist.Fit(cal_fit, "R+S")
        channels.append(cal_fit.GetParameter('Mean'))

    channels = np.array(channels, dtype='float64')
    energies = np.array(energies, dtype='float64')

    print(channels)
    print(energies)

    graph = ROOT.TGraph(len(channels), channels, energies)
    fit = ROOT.TF1("calib", "pol1", min(channels), max(channels))
    graph.Fit(fit)

    
    a = fit.GetParameter(1)
    b = fit.GetParameter(0)

    return [a, b]

def calibrated_histogram(linear_fit, acquisition, n_of_bins, name):
    a = linear_fit[0]
    b = linear_fit[1]
    data_cal = a * (acquisition['s']/len(acquisition.active_chs)) + b

    emax = a * n_of_bins + b
    emin = b

    print(emin, emax)
    hist_cal = ROOT.TH1F(name, "Calibrated Spectrum", n_of_bins, emin, emax)
    hist_cal.Fill(data_cal)

    return(hist_cal)

def calibrated_acquisition(linear_fit, acquisition):
    a = linear_fit[0]
    b = linear_fit[1]
    print(a, b)
    data_cal = a * (acquisition['s']/len(acquisition.active_chs)) + b

    emax = a * BITS12 + b
    emin = b

    return(data_cal, emin, emax)

    
                    
hist = hists['Na22_clean']
energy_ranges = [(150, 200), (320, 420), (450, 520)]
energies = Na22_MeV

c_fit = calibration_fit(hist, energy_ranges, energies)
print(c_fit)
hist_cal_bg = calibrated_histogram(c_fit, acqs[4], BITS12, "hist_cal_bg")
hist_cal_bg.Scale(1/acqs[4].exposure) 
hist_cal_Na22 = calibrated_histogram(c_fit, acqs[3], BITS12, "hist_cal_Na22")
hist_cal_Na22.Scale(1/acqs[3].exposure) 
print(acqs[3])
hist_cal_clean = hist_cal_Na22.Clone("Calibrated signal no BG")

#hist_cal = ROOT.TH1F("h_cal", "Calibrated Spectrum", BITS12, emin, emax)
#hist_cal.Fill(data_cal)


[177.38354136 371.93022564 471.39142955]
[0.511 1.275 1.786]
[0.004280158214429103, -0.2655932438428593]
-0.2655932438428593 17.265934802458748
-0.2655932438428593 17.265934802458748
SIPHRAAcquisition(File: 'SUBT_04_EnergyResolution_thr30_gain1over100_1over3_Na22')
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =       50.431
NDf                       =           47
Edm                       =  6.26293e-06
NCalls                    =          110
Constant                  =      187.956   +/-   4.04943     
Mean                      =      177.384   +/-   1.67095     
Sigma                     =      37.5292   +/-   4.35406      	 (limited)
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      152.171
NDf                       =           97
Edm                       =  2.80563e-08
NCalls                    =           96
Constant                  =      606.675   +/-   4.80363     
Mean

In [10]:
# Plot all the histograms
if ROOT.gROOT.FindObject('cv1'):
    ROOT.gROOT.FindObject('cv1').Close()
cv1 = ROOT.TCanvas("cv1", "cv1", 1600, 800)
cv1.Divide(2,1)

lg1 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
cv1.cd(1)
hist_cal_bg.SetLineColor(colors[0])
hist_cal_Na22.SetLineColor(colors[1])
lg1.AddEntry(hist_cal_bg, "Background", "l")
lg1.AddEntry(hist_cal_Na22, r"Signal ^{22}Na", "l")
hist_cal_bg.Draw("hist")
hist_cal_Na22.Draw("sames hist")
lg1.Draw()
cv1.cd(1).SetLogy()
#hist_cal_bg.GetYaxis().SetRangeUser(0,1e4)


lg2 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
cv1.cd(2)
hist_cal_clean.SetLineColor(colors[2])
hist_cal_clean.Draw("hist")
cv1.cd(2).SetLogy()

cv1.Draw()

In [20]:
def energy_Resolution(hist, peak_range):
    resolution_fit = ROOT.TF1("res_fit_" + str(i), "gaus", peak_range[0], peak_range[1])
    hist.Fit(resolution_fit, "R+S")
    std_dev = resolution_fit.GetParameter('Sigma')
    resolution = 2.355 * std_dev

    return resolution 

r = energy_Resolution(hist_cal_clean ,(1.5 , 1.9))

print(r)


0.45246703226380525
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      811.832
NDf                       =           90
Edm                       =  1.49727e-08
NCalls                    =          102
Constant                  =      2.92324   +/-   0.0161931   
Mean                      =      1.72903   +/-   0.00140192  
Sigma                     =      0.19213   +/-   0.00251407   	 (limited)
